# Simple Stamping Logic of a DC Circuit Simulator

Reference Text: Lawrence T.. Pillage, Rohrer, R. A., & Visweswariah, C. (1995). Electronic Circuit and System Simulation Methods. McGraw-Hill.

In [ ]:
import numpy as np

def init(node_n: int):
    """Initialize MNA matrix A and RHS vector J."""
    A = np.zeros((node_n, node_n), dtype=float)
    J = np.zeros((node_n, 1), dtype=float)
    return A, J


def _idx(node: int):
    """
    Convert circuit node number to Python index.
    Ground node 0 is not represented in the matrix.
    """
    return node - 1 if node > 0 else None

def solve(A: np.ndarray, J: np.ndarray):
    """Solve A x = J."""
    return np.linalg.solve(A, J).reshape(-1)

## Stamping Logic

### Resistor

- MNA Matrix $\mathbf{A}$ Stamp: $\begin{array}{c|cc}  & i & j \\ \hline i & 1/R & -1/R \\ j & -1/R & 1/R \end{array}$
> *See (Pillage et al., 1995) p7 fig 1.6a*

In [11]:

def add_r(A: np.ndarray, J: np.ndarray, i: int, j: int, r: float):
    """Stamp a resistor between node i and node j."""
    newA = A.copy()
    ii, jj = _idx(i), _idx(j)

    if ii is not None:
        newA[ii, ii] += 1.0 / r
        if jj is not None:
            newA[ii, jj] -= 1.0 / r
            newA[jj, ii] -= 1.0 / r

    if jj is not None:
        newA[jj, jj] += 1.0 / r

    newJ = J.copy()
    return newA, newJ


### Independent Current Source

Current flows from port $i$ (-) to port $j$ (+).

- For the nodes with current flowing in (like $j$ here), $\mathbf{J}$ stamp will be **positive**.
- For the nodes with current flowing out (like $i$ here), $\mathbf{J}$ stamp will be **negative**.

- RHS Bias Vector $\mathbf{J}$ stamp：$\begin{array}{c|c} \hline i & -I\\j & I \end{array}$

> *See (Pillage et al., 1995) p7 fig 1.6b*

In [ ]:

def add_constant_i(A: np.ndarray, J: np.ndarray, i: int, j: int, ii_val: float):
    """
    Stamp an independent current source. (i- +j)
      if i > 0: J[i] -= I
      if j > 0: J[j] += I
    """
    newA = A.copy()
    newJ = J.copy()

    ii, jj = _idx(i), _idx(j)

    if ii is not None:
        newJ[ii, 0] -= ii_val
    if jj is not None:
        newJ[jj, 0] += ii_val

    return newA, newJ


## Independent Voltage Source

- Positive port: $i$ (+), Negative port: $j$ (-).

- Add a new row to the MNA Matrix $\mathbf{A}$ to model the voltage difference constraint: $v_i-v_j=V_{ij}$.
- Add a new column to the MNA Matrix $\mathbf{A}$ to introduce the new current variable to the nodes' KCL equations.

$$
\left[\begin{array}{ccccc|c}
\qquad\, &  &  &            &  & \vdots \\
 &  &  &            & i\!\!\!\!\!\! & -1 \\
 &  & \mathbf{A} &  &  & \vdots \\
 &  &  &            & j\!\!\!\!\!\! & 1 \\
 & i &  &           j &  & \vdots \\\hline
\cdots & 1 & \cdots & -1 & \cdots & 0 \\
\end{array}\right]
\left[\begin{array}{c}
\vphantom{\vdots} \\
\vphantom{1} \\
 \mathbf{v}\vphantom{\vdots} \\
\vphantom{-1} \\
\vphantom{\vdots} \\\hline
I_{ji} \\
\end{array}\right]=\left[\begin{array}{c}
\vphantom{\vdots} \\
\vphantom{-1} \\
 \mathbf{J}\vphantom{\vdots} \\
\vphantom{1} \\
\vphantom{\vdots} \\\hline
V_{ij} \\
\end{array}\right]
$$

- Unknown current's direction are defined reversely ($I_{ji}$) to match "the current flowing through the voltage source" intuition.

> *See (Pillage et al., 1995) p32 (2.2.3)*

In [ ]:

def add_constant_v(A: np.ndarray, J: np.ndarray, i: int, j: int, v: float):
    """
    Stamp an independent voltage source with voltage (v_i - v_j) = v.
    This adds one extra unknown: the source branch current.
    """
    ii, jj = _idx(i), _idx(j)

    # Add one row: voltage constraint
    newA = np.vstack([A, np.zeros((1, A.shape[1]), dtype=A.dtype)])
    last_row = newA.shape[0] - 1

    if ii is not None:
        newA[last_row, ii] = 1.0
    if jj is not None:
        newA[last_row, jj] = -1.0

    # Add one column: KCL contribution of source current unknown
    newA = np.hstack([newA, np.zeros((newA.shape[0], 1), dtype=A.dtype)])
    last_col = newA.shape[1] - 1

    if ii is not None:
        newA[ii, last_col] = -1.0
    if jj is not None:
        newA[jj, last_col] = 1.0

    newJ = np.vstack([J, [[v]]])
    return newA, newJ

### Voltage Control Voltage Source (VCVS)

- Input ("Voltmeter"): Positive port: $p$ (+), Negative port: $q$ (-).
- Output: Positive port: $i$ (+), Negative port: $j$ (-).

- Add a new row to the MNA Matrix $\mathbf{A}$ to model the voltage difference constraint: $v_i-v_j=g(v_p-v_q)$.
- Add a new column to the MNA Matrix $\mathbf{A}$ to introduce the new current variable to the nodes' KCL equations. (same as the independent voltage sources)

In [ ]:
def add_v_ctrl_v(
    A: np.ndarray, J: np.ndarray, p: int, q: int, i: int, j: int, g: float
):
    """
    Stamp a VCVS:
        (v_i - v_j) = g * (v_p - v_q)
    Adds one extra unknown: the VCVS branch current.
    """
    pp, qq = _idx(p), _idx(q)
    ii, jj = _idx(i), _idx(j)

    # Add one row: controlled voltage constraint
    newA = np.vstack([A, np.zeros((1, A.shape[1]), dtype=A.dtype)])
    last_row = newA.shape[0] - 1

    if ii is not None:
        newA[last_row, ii] = 1.0
    if jj is not None:
        newA[last_row, jj] = -1.0
    if pp is not None:
        newA[last_row, pp] = -g
    if qq is not None:
        newA[last_row, qq] = g

    # Add one column: KCL contribution of source current unknown
    newA = np.hstack([newA, np.zeros((newA.shape[0], 1), dtype=A.dtype)])
    last_col = newA.shape[1] - 1

    if ii is not None:
        newA[ii, last_col] = -1.0
    if jj is not None:
        newA[jj, last_col] = 1.0

    newJ = np.vstack([J, [[0.0]]])
    return newA, newJ

### Simple NPN BJT Model

- Modeled as an constant voltage drop diode (0.7V) plus a Current Control Current Source (CCCS) with gain $\beta$.

<center><img src="assets/npnbjt_model.png" width=300 /></center>
<center><i>source: EE2027 AY2526S2 #BJT-17</i></center>

- Utilize the current variable added when stamping the voltage drop part to implement the CCCS.

In [ ]:

def add_npn_bjt(A: np.ndarray, J: np.ndarray, b: int, c: int, e: int, beta: float):
    """
    Very simple NPN BJT model:
    1) base-emitter constant voltage drop of 0.7V
    2) CCCS with gain beta using the added voltage-source current as control current
    """
    newA, newJ = add_constant_v(A, J, b, e, 0.7)

    last_col = newA.shape[1] - 1
    ee, cc = _idx(e), _idx(c)

    if ee is not None:
        newA[ee, last_col] += beta
    if cc is not None:
        newA[cc, last_col] -= beta

    return newA, newJ

## Tests

### Test 1: Voltage Divider
<center><img src="assets/test1_voltage_divider.png" width=400 /></center>

In [24]:
# Test 1: Voltage Divider
A, J = init(2)
A, J = add_constant_v(A, J, 1, 0, 5)
A, J = add_r(A, J, 1, 2, 3)
A, J = add_r(A, J, 2, 0, 2)
print("Test 1:", solve(A, J))
# expected: [5.0, 2.0, 1.0]

Test 1: [5. 2. 1.]


### Test 2: Complex Circuit with R and Voltage Source

<center><img src="assets/test2.png" width=300 /></center>
<center><i>source: EE1111A AY2223S1 Quiz 2 #16</i></center>
<center><img src="assets/test2_1.png" width=300 /></center>

In [25]:
# Test 2: Complex Circuit with R and Voltage Source
A, J = init(4)
A, J = add_constant_v(A, J, 1, 0, 1)
A, J = add_r(A, J, 1, 2, 50)
A, J = add_r(A, J, 2, 3, 50)
A, J = add_r(A, J, 3, 4, 50)
A, J = add_r(A, J, 3, 0, 27.5)
A, J = add_r(A, J, 2, 4, 90.92)
print("Test 2:", solve(A, J))
# expected approx: [1.0000, 0.5630, 0.2404, 0.3548, 0.0087]

Test 2: [1.         0.56295811 0.24037304 0.35482985 0.00874084]


### Test 3: Current Source

<center><img src="assets/test3.png" width=300 /></center>

In [19]:
# Test 3: Current Source
A, J = init(3)
A, J = add_constant_i(A, J, 1, 2, 0.02)
A, J = add_constant_i(A, J, 2, 3, 0.06)
A, J = add_r(A, J, 1, 2, 1300)
A, J = add_r(A, J, 2, 3, 1000)
A, J = add_r(A, J, 3, 0, 2000)
A, J = add_r(A, J, 2, 0, 1500)
print("Test 3:", solve(A, J))
# expected approx: [-46, -20, 26.67]

Test 3: [-46.         -20.          26.66666667]


### Test 4: VCVS

<center><img src="assets/test4.png" width=400 /></center>

In [20]:
# Test 4: VCVS
A, J = init(5)
A, J = add_constant_v(A, J, 1, 0, 5)
A, J = add_constant_v(A, J, 2, 0, 3)
A, J = add_r(A, J, 1, 3, 1)
A, J = add_r(A, J, 2, 4, 2)
A, J = add_r(A, J, 5, 0, 8)
A, J = add_v_ctrl_v(A, J, 3, 4, 5, 0, 4)
print("Test 4:", solve(A, J))
# expected approx: [5, 3, 5, 3, 8, 0, 0, 1]

Test 4: [ 5.  3.  5.  3.  8. -0. -0.  1.]


### Test 5: Inverting Amplifier

<center><img src="assets/test5.png" width=400 /></center>

In [21]:
# Test 5: Inverting Amplifier
A, J = init(3)
A, J = add_constant_v(A, J, 1, 0, 10)
A, J = add_r(A, J, 1, 2, 3)
A, J = add_r(A, J, 2, 3, 2)
A, J = add_v_ctrl_v(A, J, 2, 0, 3, 0, 1_000_000)
print("Test 5:", solve(A, J))
# expected approx: [10 0 -6.67 3.33 -3.33]
# node 2 will be very close to 0, not exactly 0, because gain is finite

Test 5: [ 1.00000000e+01 -6.66667778e-06 -6.66667778e+00  3.33333556e+00
 -3.33333556e+00]


### Test 6: NPN BJT DC Circuit

<center><img src="assets/test6.png" width=400 /></center>
<center><i>source: EE2027 AY2526S2 #BJT-19</i></center>
<center><img src="assets/test6_1.png" width=300 /></center>

In [22]:
# Test 6: NPN BJT (CCCS)
A, J = init(4)
A, J = add_constant_v(A, J, 1, 0, 3.7)
A, J = add_constant_v(A, J, 4, 0, 10)
A, J = add_npn_bjt(A, J, 2, 3, 0, 100)
A, J = add_r(A, J, 1, 2, 10000)
A, J = add_r(A, J, 3, 4, 220)
print("Test 6:", solve(A, J))
# expected approx: [3.7, 0.7, 3.4, 10.0, 0.0003, 0.0300, -0.0003]

Test 6: [ 3.7e+00  7.0e-01  3.4e+00  1.0e+01  3.0e-04  3.0e-02 -3.0e-04]
